<a href="https://colab.research.google.com/github/sachitkuhar/ML_Coding/blob/main/DD_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import torch
import torch.nn.functional as F
import math

In [28]:
def stable_softmax(x, dim=-1):
    # softmax(x)_i = exp(x_i - max) / sum_j exp(x_j - max)   -- subtract max for stability
    # keepdim=True on both reductions so they broadcast back over `dim`
    max_x, _ = torch.max(x, dim=dim, keepdim=True)
    exp_x = torch.exp(x - max_x)
    y = torch.sum(exp_x, dim=dim, keepdim=True)
    softmax_x = exp_x / y
    return softmax_x


def stable_log_softmax(x, dim=-1):
    # log_softmax(x) = x - logsumexp(x, dim, keepdim=True)
    # return torch.log_softmax(x, dim=dim)
    return x - x.logsumexp(dim, keepdim=True)
    ...

In [29]:
def _test_softmax():
    x = torch.randn(4,6)
    assert torch.allclose(stable_softmax(x), x.softmax(-1), atol=1e-6)
    assert torch.allclose(stable_log_softmax(x), F.log_softmax(x,-1), atol=1e-6)
    assert torch.allclose(stable_softmax(x).sum(-1), torch.ones(4), atol=1e-6)
    assert torch.allclose(stable_softmax(x, dim=0).sum(0), torch.ones(6), atol=1e-6)  # dim arg works
    assert stable_softmax(torch.tensor([[1000.,1001.,1002.]])).isfinite().all()        # no overflow
    assert torch.allclose(stable_log_softmax(x).exp(), stable_softmax(x), atol=1e-6)
    print("✓ softmax")
_test_softmax()

✓ softmax


In [34]:
def cross_entropy(logits, targets, ignore_index=-100, label_smoothing=0.0, reduction='mean'):
    # logits (N,V), targets (N,). nll_i = -log_softmax(logits_i)[target_i]
    # label smoothing: loss_i = (1-eps)*nll_i + eps * mean_over_vocab(-log_softmax_i)
    # ignore_index: drop those positions from the loss AND the mean denominator
    # trick: targets.clamp_min(0) before gather so a -100 index doesn't crash; mask them out after
    # reduction in {'none','sum','mean}; 'mean' divides by count of valid tokens (clamp_min(1))
    ...
    logp = torch.log_softmax(logits, dim=-1)
    nll = -logp.gather(-1, targets.clamp_min(0).unsqueeze(-1)).squeeze(-1)
    smooth = - logp.mean(dim=-1)
    loss = (1 - label_smoothing) * nll + label_smoothing * smooth
    mask = (targets != ignore_index).float()
    loss = loss * mask
    if reduction == 'none':
        return loss
    if reduction == 'sum':
        return loss.sum()
    return loss.sum() / mask.sum().clamp_min(1)                             # divide by valid count only    # max_x,_ = torch.max(logits, dim=-1, keepdim=True)
    # exp_x = torch.exp(logits - max_x)
    # sum_exp_x = torch.sum(exp_x, dim=-1)
    # ce_loss = torch.log(sum_exp_x) - torch.log(exp_x.gather(-1, targets.unsqueeze(-1)).unsqueeze(-1))
    # return ce_loss.mean()

In [35]:
def _test_cross_entropy():
    logits = torch.randn(5,7); targets = torch.randint(0,7,(5,))
    assert torch.allclose(cross_entropy(logits,targets), F.cross_entropy(logits,targets), atol=1e-5)
    t2 = targets.clone(); t2[0]=t2[2]=-100
    assert torch.allclose(cross_entropy(logits,t2,ignore_index=-100),
                          F.cross_entropy(logits,t2,ignore_index=-100), atol=1e-5)
    assert torch.allclose(cross_entropy(logits,targets,label_smoothing=0.1),
                          F.cross_entropy(logits,targets,label_smoothing=0.1), atol=1e-5)
    assert torch.allclose(cross_entropy(logits,targets,reduction='none'),
                          F.cross_entropy(logits,targets,reduction='none'), atol=1e-5)
    assert cross_entropy(logits, torch.full((5,),-100), ignore_index=-100) == 0        # no nan
    assert torch.allclose(cross_entropy(torch.zeros(3,7), torch.tensor([0,1,2])),
                          torch.tensor(math.log(7)), atol=1e-5)                          # uniform -> log V
    print("✓ cross_entropy")
_test_cross_entropy()

✓ cross_entropy


In [38]:
def sdpa(Q, K, V, attn_mask=None, is_causal=False):
    # Q,K,V: (..., L, d). scores = Q @ K^T / sqrt(d)  -> (..., Lq, Lk)
    # is_causal: block j>i with an upper-triangular mask (torch.triu(..., diagonal=1))
    # attn_mask: bool, True = BLOCK. masked_fill blocked scores with -inf BEFORE softmax
    # softmax over the LAST dim (keys). return (output (..., Lq, d), attn (..., Lq, Lk))
    ...
    d = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / d**0.5
    if is_causal:
      Lq, Lk = scores.shape[-2], scores.shape[-1]
      causal = torch.triu(torch.ones(Lq, Lk, dtype=torch.bool, device=scores.device), diagonal=1)
      scores = scores.masked_fill(causal, float('-inf'))
    if attn_mask is not None:
      scores = scores.masked_fill(attn_mask, float('-inf'))
    attn = scores.softmax(-1)
    return attn @ V, attn

In [39]:
def _test_sdpa():
    Q,K,V = torch.randn(2,3,5,8), torch.randn(2,3,5,8), torch.randn(2,3,5,8)
    out,_ = sdpa(Q,K,V)
    assert torch.allclose(out, F.scaled_dot_product_attention(Q,K,V), atol=1e-5)
    outc,attn = sdpa(Q,K,V,is_causal=True)
    assert torch.allclose(outc, F.scaled_dot_product_attention(Q,K,V,is_causal=True), atol=1e-5)
    assert torch.allclose(attn.sum(-1), torch.ones(2,3,5), atol=1e-6)              # rows sum to 1
    assert torch.allclose(attn[:,:,0,1:], torch.zeros(2,3,4), atol=1e-6)          # causal: tok0 -> only tok0
    pad = torch.zeros(2,1,1,5,dtype=torch.bool); pad[...,3:]=True                  # block keys 3,4 (broadcasts over heads/queries)
    _,ap = sdpa(Q,K,V,attn_mask=pad)
    assert torch.allclose(ap[...,3:], torch.zeros(2,3,5,2), atol=1e-6)
    print("✓ sdpa")
_test_sdpa()

✓ sdpa


In [42]:
def rmsnorm(x, weight, eps=1e-6):
    # x / sqrt(mean(x^2, last dim) + eps) * weight   (no mean subtraction)
    ...
    rms_x = torch.sqrt(torch.mean(torch.pow(x, 2), dim=-1, keepdim=True) + eps)
    output = x / rms_x * weight
    return output

def layernorm(x, weight, bias, eps=1e-5):
    # (x - mean) / sqrt(var + eps) * weight + bias, over the LAST dim
    # var is the BIASED estimate (unbiased=False, divide by N); eps INSIDE the sqrt
    ...
    mean = torch.mean(x, dim=-1, keepdim=True)
    var = torch.var(x, dim=-1, unbiased=False, keepdim=True)
    return ((x - mean)/torch.sqrt(var + eps)) * weight + bias

In [43]:
def _test_norm():
    x=torch.randn(4,10); w=torch.randn(10); b=torch.randn(10)
    assert torch.allclose(layernorm(x,w,b), F.layer_norm(x,(10,),w,b), atol=1e-5)
    ln = layernorm(x, torch.ones(10), torch.zeros(10))
    assert ln.mean(-1).abs().max()<1e-5 and (ln.std(-1,unbiased=False)-1).abs().max()<1e-4
    assert torch.allclose(rmsnorm(x,w), F.rms_norm(x,(10,),w), atol=1e-5)   # if no F.rms_norm, compare to manual
    assert rmsnorm(x,w).shape == x.shape
    print("✓ norm")
_test_norm()

✓ norm


In [50]:
def filter_logits(logits, top_k=0, top_p=1.0, temperature=1.0):
    # 1) divide logits by temperature
    # 2) top_k>0: mask everything below the k-th largest value to -inf
    # 3) top_p<1: sort desc, cumsum of softmax; remove tokens past p but SHIFT so the
    #    crossing token is kept, and never drop the top-1; scatter the removal mask back
    #    to original order before masked_fill
    ...
    logits = logits / temperature
    if top_k > 0:
      k = min(top_k, logits.shape[-1])
      threshold = torch.topk(logits, k, dim=-1).values[...,-1:]
      logits = logits.masked_fill(logits < threshold, float('-inf'))
    if top_p < 1.0:
      prob = logits.softmax(dim=-1)
      sorted_p, indices_p = torch.sort(prob, dim=-1, descending=True)
      cumsum = torch.cumsum(sorted_p, dim=-1)
      sorted_mask = (cumsum - sorted_p) >= top_p
      mask = torch.zeros_like(sorted_p, dtype=torch.bool).scatter(-1, indices_p, sorted_mask)
      logits = logits.masked_fill(mask, float('-inf'))
    return logits
    # probs = torch.softmax(logits, dim=-1)
    # samples = torch.multinomial(prob, num_samples=1)


In [51]:
def _test_filter():
    lg = torch.tensor([[2.,1.,0.5,-1.,-3.]])
    assert torch.allclose(filter_logits(lg.clone(), temperature=2.0), lg/2.0)
    assert torch.isfinite(filter_logits(lg.clone(), top_k=2)).sum() == 2
    kept = torch.isfinite(filter_logits(lg.clone(), top_p=0.8))
    assert kept[0].tolist() == [True,True,False,False,False]              # crossing token kept
    assert torch.isfinite(filter_logits(lg.clone(), top_p=0.01)).sum() >= 1   # always keep >=1
    assert torch.allclose(filter_logits(lg.clone(), top_p=0.8).softmax(-1).sum(), torch.tensor(1.0))
    print("✓ filter")
_test_filter()

✓ filter


In [52]:
def squared_dist(A, B):
    # A (N,D), B (M,D) -> (N,M). ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a.b
    # clamp_min(0) the result (float error can make it slightly negative)
    ...
    return (A.pow(2).sum(-1, keepdim=True) + B.pow(2).sum(-1) - 2 * A @ B.T).clamp_min(0)

def cosine_sim(A, B, eps=1e-8):
    # normalize each row (clamp_min(eps) the norm so a zero vector doesn't divide by 0), then A_n @ B_n^T
    ...
    An = A / A.norm(dim=-1, keepdim=True).clamp_min(eps)
    Bn = B / B.norm(dim=-1, keepdim=True).clamp_min(eps)
    return An @ Bn.T

In [53]:
def _test_pairwise():
    A=torch.randn(4,5); B=torch.randn(6,5)
    assert torch.allclose(squared_dist(A,B), torch.cdist(A,B)**2, atol=1e-4)
    assert squared_dist(A,A).diag().abs().max() < 1e-4                    # self-distance ~0
    assert (squared_dist(A,B) >= 0).all()
    cs = cosine_sim(A,B)
    man = torch.stack([torch.stack([F.cosine_similarity(a,b,dim=0) for b in B]) for a in A])
    assert torch.allclose(cs, man, atol=1e-5)
    assert (cs.abs() <= 1+1e-5).all()
    assert cosine_sim(torch.zeros(1,5), B).isfinite().all()               # zero vector, no nan
    print("✓ pairwise")
_test_pairwise()

✓ pairwise


In [54]:
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1):
    # all (B,): per-sequence summed log-probs of chosen/rejected under policy and ref
    # logits = beta * ((policy_chosen - ref_chosen) - (policy_rejected - ref_rejected))
    # loss   = -mean(logsigmoid(logits))   -- use F.logsigmoid; DETACH the ref terms
    logits = beta * ((policy_chosen - ref_chosen.detach())
                     - (policy_rejected - ref_rejected.detach()))   # two log-ratios vs ref
    return -F.logsigmoid(logits).mean()                             # logsigmoid, not log(sigmoid)

In [55]:
def _test_dpo():
    pc,pr,rc,rr = torch.randn(8),torch.randn(8),torch.randn(8),torch.randn(8)
    assert torch.allclose(dpo_loss(pc,pr,rc,rr), -F.logsigmoid(0.1*((pc-rc)-(pr-rr))).mean(), atol=1e-6)
    z=torch.zeros(4)
    assert torch.allclose(dpo_loss(z,z,z,z), torch.tensor(math.log(2)), atol=1e-6)   # equal margins -> log2
    lo = dpo_loss(torch.ones(4)*5, torch.zeros(4), torch.zeros(4), torch.zeros(4))
    hi = dpo_loss(torch.zeros(4), torch.ones(4)*5, torch.zeros(4), torch.zeros(4))
    assert lo < hi                                                                    # bigger chosen margin -> lower loss
    rcg=torch.randn(8,requires_grad=True); pcg=torch.randn(8,requires_grad=True)
    dpo_loss(pcg,pr,rcg,rr).backward()
    assert rcg.grad is None and pcg.grad is not None                                  # ref detached
    print("✓ dpo")
_test_dpo()

✓ dpo


In [56]:
def top_k_accuracy(logits, targets, k):
    # logits (N,V), targets (N,). fraction of rows where target is among the top-k logits.
    # topk indices (N,k); compare to targets.unsqueeze(-1); .any(-1); mean of the float
    ...
    topk = logits.topk(k, dim=-1).indices
    correct = (topk == targets.unsqueeze(-1)).any(-1)
    return correct.float().mean()

In [57]:
def _test_topk_acc():
    logits=torch.randn(20,10); targets=torch.randint(0,10,(20,))
    assert torch.allclose(top_k_accuracy(logits,targets,1), (logits.argmax(-1)==targets).float().mean())
    assert top_k_accuracy(logits,targets,10) == 1.0
    lg=torch.tensor([[0.,1.,2.]])
    assert top_k_accuracy(lg,torch.tensor([1]),2)==1.0 and top_k_accuracy(lg,torch.tensor([0]),2)==0.0
    print("✓ topk_acc")
_test_topk_acc()

✓ topk_acc
